# Pretrained ViT Action Recognition

This notebook fine-tunes a pretrained Vision Transformer (ViT) for action classification using KerasHub. The pretrained model includes its matching image preprocessor.

## NVIDIA GPU setup on Windows

TensorFlow 2.11 and newer do not use CUDA GPUs from a native Windows Python environment. To use the NVIDIA GPU, run this notebook in WSL2:

1. Install WSL2 and Ubuntu from an elevated PowerShell with `wsl --install -d Ubuntu`, then restart Windows if prompted.
2. Install the VS Code WSL extension and open this project folder in WSL (`/mnt/d/LIME`). Select a Python kernel created inside WSL, not the Windows `.venv`.
3. Run the install cell below in the WSL kernel. The Windows NVIDIA driver is used by WSL2; do not install a separate Linux NVIDIA display driver inside WSL.
4. Run the GPU setup cell. It stops with an error if TensorFlow cannot see and execute work on the GPU. ViT-B/16 is larger than the CNN models, so training uses batches of 8 images.

In [ ]:
import platform
import subprocess
import sys

if platform.system() != 'Linux':
    raise RuntimeError(
        'Use a Linux/WSL2 Python kernel for NVIDIA CUDA support; native Windows TensorFlow cannot use this GPU.'
    )

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'tensorflow[and-cuda]', 'keras-hub', 'ipykernel', 'scikit-learn', 'matplotlib', 'pillow', 'numpy', 'opencv-python'
])

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tensorflow import keras
from tensorflow.keras import layers
import keras_hub
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf

print(f'TensorFlow version: {tf.__version__}')
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")

In [ ]:
gpu_devices = tf.config.list_physical_devices('GPU')

if not gpu_devices:
    raise RuntimeError(
        'No TensorFlow GPU found. Use the WSL2 Linux kernel and install tensorflow[and-cuda] as described above.'
    )

for gpu in gpu_devices:
    tf.config.experimental.set_memory_growth(gpu, True)

with tf.device('/GPU:0'):
    gpu_probe = tf.linalg.matmul(tf.ones((1, 1)), tf.ones((1, 1)))
if 'GPU:0' not in gpu_probe.device:
    raise RuntimeError(f'TensorFlow did not place its test operation on the GPU: {gpu_probe.device}')

print(f'GPU ready: {gpu_devices[0].name}')

In [ ]:
# Dataset paths
dataset_root_candidates = [
    r'/mnt/d/LIME/VOCdevkit/VOC2012',
    r'/mnt/m/LIME/VOCdevkit/VOC2012',
    r'D:/LIME/VOCdevkit/VOC2012',
    r'm:/LIME/VOCdevkit/VOC2012',
    r'D:\LIME\VOCdevkit\VOC2012',
    r'm:\LIME\VOCdevkit\VOC2012'
]
dataset_root = next((p for p in dataset_root_candidates if os.path.exists(p)), dataset_root_candidates[0])

image_dir = os.path.join(dataset_root, 'JPEGImages')
action_dir = os.path.join(dataset_root, 'ImageSets', 'Action')
print(f'Dataset root: {dataset_root}')
print(f'Image directory: {image_dir}')
print(f'Action directory: {action_dir}')

action_classes = [
    'jumping', 'phoning', 'playinginstrument', 'reading', 'ridingbike',
    'ridinghorse', 'running', 'takingphoto', 'usingcomputer', 'walking'
]
print('Action classes:', action_classes)

In [ ]:
def load_action_dataset(action_classes, image_dir, action_dir, target_size=(224, 224), limit_per_class=None):
    if not os.path.isdir(image_dir):
        raise FileNotFoundError(f'VOC image directory not found: {image_dir}')
    if not os.path.isdir(action_dir):
        raise FileNotFoundError(f'VOC Action annotation directory not found: {action_dir}')

    images = []
    labels = []

    for class_index, class_name in enumerate(action_classes):
        train_file = os.path.join(action_dir, f'{class_name}_train.txt')
        val_file = os.path.join(action_dir, f'{class_name}_val.txt')
        annotation_files = [path for path in (train_file, val_file) if os.path.isfile(path)]
        if not annotation_files:
            raise FileNotFoundError(f'No train/validation annotation files found for {class_name} in {action_dir}')

        seen = set()
        class_sample_count = 0
        for annotation_file in annotation_files:
            if limit_per_class is not None and class_sample_count >= limit_per_class:
                break
            with open(annotation_file, 'r', encoding='utf-8') as f:
                for line_number, line in enumerate(f, start=1):
                    fields = line.split()
                    if not fields:
                        continue
                    if len(fields) < 3:
                        raise ValueError(
                            f'Expected image id, instance id, and label in {annotation_file}:{line_number}; '
                            f'got {line.rstrip()!r}'
                        )

                    image_id, _, class_label = fields[:3]
                    if class_label != '1' or image_id in seen:
                        continue
                    seen.add(image_id)

                    img_path = os.path.join(image_dir, f'{image_id}.jpg')
                    if not os.path.isfile(img_path):
                        continue

                    img = load_img(img_path, target_size=target_size, color_mode='rgb')
                    arr = img_to_array(img)
                    # Keep RGB values raw (0-255); the KerasHub task applies preset preprocessing in fit() and predict().
                    images.append(arr)
                    labels.append(class_index)
                    class_sample_count += 1

                    if limit_per_class is not None and class_sample_count >= limit_per_class:
                        break

    if not images:
        raise ValueError(
            f'No positive VOC Action samples were loaded from {action_dir}. '
            'Check that the annotation files contain positive labels and matching JPEG images exist.'
        )

    return np.array(images, dtype='float32'), np.array(labels, dtype='int32')

X, y = load_action_dataset(action_classes, image_dir, action_dir, limit_per_class=None)
print(f'Loaded {len(X)} positive training samples across {len(action_classes)} action classes')
print(f'Example image shape: {X[0].shape}')
print('Label distribution:', {cls: int(np.sum(y == i)) for i, cls in enumerate(action_classes)})

In [ ]:
# Split into train/validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


print(f'Train samples: {len(X_train)}')
print(f'Validation samples: {len(X_val)}')

In [ ]:
# Load pretrained ViT-B/16 weights and replace its ImageNet head for the VOC action classes.
with tf.device('/GPU:0'):
    vit_model = keras_hub.models.ViTImageClassifier.from_preset(
        'vit_base_patch16_224',
        num_classes=len(action_classes),
        activation=None,
        compile=False
    )
    vit_model.backbone.trainable = False
    vit_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )

vit_model.summary()

In [ ]:
# Augment raw RGB images; the KerasHub task applies its preset preprocessing in fit() and predict().
batch_size = 8
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.055),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomZoom(0.1),
    layers.RandomFlip('horizontal')
], name='data_augmentation')

train_dataset = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(buffer_size=len(X_train))
    .batch(batch_size)
    .map(
        lambda images, labels: (data_augmentation(images, training=True), labels),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    .prefetch(tf.data.AUTOTUNE)
)
val_dataset = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

history = vit_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    verbose=1
)

vit_model.save('pretrained_vit_action_model.keras')
print('Saved ViT model to pretrained_vit_action_model.keras')

In [ ]:
# Evaluate the trained ViT model
val_predictions = vit_model.predict(X_val, batch_size=8, verbose=0)
val_pred_labels = np.argmax(val_predictions, axis=1)

print(classification_report(y_val, val_pred_labels, target_names=action_classes))
print('Confusion matrix:')
print(confusion_matrix(y_val, val_pred_labels))

# Show predicted class for a few sample images
sample_indices = np.random.choice(len(X_val), size=5, replace=False)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, idx in zip(axes, sample_indices):
    image_arr = np.clip(X_val[idx], 0, 255).astype(np.uint8)
    pred_idx = int(np.argmax(val_predictions[idx]))
    actual_idx = int(y_val[idx])
    ax.imshow(image_arr)
    ax.set_title(f'Pred: {action_classes[pred_idx]}\nActual: {action_classes[actual_idx]}', fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Summary

This notebook uses a Pretrained ViT transfer-learning pipeline. The same dataset and image-generation process are kept, but the model is trained using a pretrained ViT backbone for image recognition and action classification.